# Experiment 1.3.10 — Fixed250+Linear vs stacked-bin SNN ablation

This notebook is **analysis-only**. SNN training and the paired Fixed250+Linear baseline are produced by the repository Slurm scripts.

Primary baseline: `[256,30] -> 16 fixed 250 ms bins -> per-channel count -> [16,30] -> flatten 480D -> train-only StandardScaler -> LogisticRegression(lbfgs)`, matching Experiment 1.3.4.

Stacked SNN input: `[256,30] -> [16 bins,16 within-bin steps,30] -> [16,480]`. All SNN conditions use whole-output-spike-count inference.

SNN factorial design: **5 architectures** (`1h128`, `1h256`, `1h512`, `1h1024`, `2h128`) × 2 objectives × 2 dynamics regimes × 3 user-disjoint split seeds = **60 SNN runs**, plus 3 paired Fixed250+Linear runs.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'snn').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate writingRing repository root')

REPO_ROOT = find_repo_root()
EXPERIMENT_ID = 'experiment_1_3_10_stacked_bin_snn_ablation'
PROTOCOL_VERSION = 'stacked250_v1'
LABELS = tuple(sorted(('A','B','C','D','E','X','G','H','I','J','K','L')))
LABEL_TAG = 'labels_' + '-'.join(LABELS)
RESULTS_DIR = REPO_ROOT / 'notebooks' / 'artifacts' / EXPERIMENT_ID / PROTOCOL_VERSION / LABEL_TAG
RUNS_DIR = RESULTS_DIR / 'runs'
HISTORIES_DIR = RESULTS_DIR / 'histories'
BASELINE_DIR = RESULTS_DIR / 'baselines'
FIGURES_DIR = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_SEEDS = (11,23,101)
ARCHITECTURES = ('1h128','1h256','1h512','1h1024','2h128')
OBJECTIVES = ('whole_count_ce','timestep_ce')
TRAIN_REGIMES = ('weight_only','trainable_dynamics')
EXPECTED_RUNS = 60
EXPECTED_BASELINES = 3
print('Results:', RESULTS_DIR)

## Load SNN and Fixed250+Linear artifacts; verify pairing

In [ ]:
def flatten_run(p):
    row = {
        'run_key': p['run_key'], 'split_seed': int(p['split_seed']),
        'architecture': p['architecture'], 'objective': p['objective'],
        'train_regime': p['train_regime'], 'best_epoch': int(p['best_epoch']),
        'model_seed': int(p['model_seed']), 'sample_hash': p['sample_hash'],
        'trainable_parameters': int(p['parameter_counts']['trainable']),
        'history_file': p['history_file'],
    }
    for split in ('train','val','test'):
        m = p['metrics'][split]
        for key in ('loss','balanced_accuracy','accuracy','macro_f1','output_firing_rate'):
            row[f'{split}_{key}'] = float(m[key])
        for i, fr in enumerate(m.get('hidden_firing_rates', []), 1):
            row[f'{split}_hidden{i}_firing_rate'] = float(fr)
    for hidden in p['best_dynamics']['hidden']:
        layer = hidden['layer']
        for q in ('tau_syn_ms','tau_mem_ms','threshold'):
            row[f'{layer}_{q}_mean'] = float(hidden[q]['mean'])
            row[f'{layer}_{q}_std'] = float(hidden[q]['std'])
    return row

def flatten_baseline(p):
    row = {'baseline_id': p['baseline_id'], 'split_seed': int(p['split_seed']), 'sample_hash': p['sample_hash']}
    for split in ('train','val','test'):
        m = p['metrics'][split]
        for key in ('balanced_accuracy','accuracy','macro_f1'):
            row[f'{split}_{key}'] = float(m[key])
    return row

run_files = sorted(RUNS_DIR.glob('run_*.json')) if RUNS_DIR.exists() else []
payloads = [json.loads(p.read_text()) for p in run_files]
runs = pd.DataFrame(flatten_run(p) for p in payloads)
if not runs.empty:
    runs = runs.sort_values(['split_seed','architecture','train_regime','objective']).reset_index(drop=True)
baseline_files = sorted(BASELINE_DIR.glob('fixed250_count_linear_split*.json')) if BASELINE_DIR.exists() else []
baseline_payloads = [json.loads(p.read_text()) for p in baseline_files]
baseline = pd.DataFrame(flatten_baseline(p) for p in baseline_payloads)
if not baseline.empty:
    baseline = baseline.sort_values('split_seed').reset_index(drop=True)
print(f'SNN completed: {len(runs)}/{EXPECTED_RUNS}')
print(f'Fixed250+Linear completed: {len(baseline)}/{EXPECTED_BASELINES}')
display(runs.head())
display(baseline)
if not runs.empty:
    assert (runs.groupby('split_seed').sample_hash.nunique() == 1).all(), 'SNN split pairing failed'
    assert (runs.groupby(['split_seed','architecture']).model_seed.nunique() == 1).all(), 'Initialization pairing failed'
if len(baseline) == EXPECTED_BASELINES and not runs.empty:
    snn_hash = runs.groupby('split_seed', as_index=False).sample_hash.first().rename(columns={'sample_hash':'snn_sample_hash'})
    paired_hash = baseline.merge(snn_hash, on='split_seed', how='inner')
    assert len(paired_hash) == EXPECTED_BASELINES
    assert (paired_hash.sample_hash == paired_hash.snn_sample_hash).all(), 'Baseline/SNN sample pairing failed'

## Aggregate performance across user splits

In [ ]:
if runs.empty:
    summary = pd.DataFrame()
else:
    summary = (runs.groupby(['architecture','train_regime','objective'], as_index=False)
        .agg(n_splits=('split_seed','nunique'),
             mean_val_ba=('val_balanced_accuracy','mean'), sd_val_ba=('val_balanced_accuracy','std'),
             mean_test_ba=('test_balanced_accuracy','mean'), sd_test_ba=('test_balanced_accuracy','std'),
             mean_test_accuracy=('test_accuracy','mean'), sd_test_accuracy=('test_accuracy','std'),
             mean_test_macro_f1=('test_macro_f1','mean'), sd_test_macro_f1=('test_macro_f1','std'),
             mean_best_epoch=('best_epoch','mean')))
if baseline.empty:
    baseline_summary = pd.DataFrame()
else:
    baseline_summary = pd.DataFrame([{
        'method':'Fixed250 + Linear', 'n_splits':baseline.split_seed.nunique(),
        'mean_val_ba':baseline.val_balanced_accuracy.mean(), 'sd_val_ba':baseline.val_balanced_accuracy.std(),
        'mean_test_ba':baseline.test_balanced_accuracy.mean(), 'sd_test_ba':baseline.test_balanced_accuracy.std(),
        'mean_test_accuracy':baseline.test_accuracy.mean(), 'sd_test_accuracy':baseline.test_accuracy.std(),
        'mean_test_macro_f1':baseline.test_macro_f1.mean(), 'sd_test_macro_f1':baseline.test_macro_f1.std(),
    }])
display(baseline_summary)
display(summary)
if not summary.empty: summary.to_csv(RESULTS_DIR/'notebook_summary.csv',index=False)
if not baseline_summary.empty: baseline_summary.to_csv(RESULTS_DIR/'notebook_fixed250_linear_summary.csv',index=False)

## Train objective loss vs epoch

In [ ]:
history_rows=[]
for p in payloads:
    history_path = REPO_ROOT / p['history_file']
    if not history_path.exists():
        continue
    h = pd.read_csv(history_path)
    h['split_seed'] = int(p['split_seed'])
    h['architecture'] = p['architecture']
    h['objective'] = p['objective']
    h['train_regime'] = p['train_regime']
    history_rows.append(h)
histories = pd.concat(history_rows, ignore_index=True) if history_rows else pd.DataFrame()
if not histories.empty:
    loss_summary=(histories.groupby(['epoch','architecture','objective','train_regime'],as_index=False)
        .agg(mean_train_objective_loss=('train_objective_loss','mean'),
             sd_train_objective_loss=('train_objective_loss','std'),
             n_splits=('split_seed','nunique')))
    display(loss_summary.head())
    loss_summary.to_csv(RESULTS_DIR/'train_loss_by_epoch_summary.csv',index=False)
    for objective in OBJECTIVES:
        fig, axes = plt.subplots(1,2,figsize=(14,5),sharey=True)
        for ax, regime in zip(axes, TRAIN_REGIMES):
            for architecture in ARCHITECTURES:
                part=loss_summary[(loss_summary.objective==objective)&(loss_summary.train_regime==regime)&(loss_summary.architecture==architecture)].sort_values('epoch')
                if part.empty: continue
                ax.plot(part.epoch,part.mean_train_objective_loss,label=architecture)
                lo=part.mean_train_objective_loss-part.sd_train_objective_loss.fillna(0)
                hi=part.mean_train_objective_loss+part.sd_train_objective_loss.fillna(0)
                ax.fill_between(part.epoch,lo,hi,alpha=.12)
            ax.set_title(regime); ax.set_xlabel('Epoch'); ax.grid(alpha=.25)
        axes[0].set_ylabel('Train objective loss')
        axes[1].legend(title='Architecture')
        fig.suptitle(f'Train loss vs epoch — {objective} (mean ± SD across splits)')
        fig.tight_layout()
        fig.savefig(FIGURES_DIR/f'train_loss_{objective}.png',dpi=180,bbox_inches='tight')
        plt.show()

## Main comparison: Fixed250+Linear vs SNN architecture × objective

In [ ]:
if not summary.empty:
    x=np.arange(len(ARCHITECTURES))
    fig,axes=plt.subplots(1,2,figsize=(15,5),sharey=True)
    baseline_mean=baseline.test_balanced_accuracy.mean() if not baseline.empty else np.nan
    baseline_sd=baseline.test_balanced_accuracy.std() if not baseline.empty else np.nan
    for ax,regime in zip(axes,TRAIN_REGIMES):
        for objective in OBJECTIVES:
            part=summary[(summary.train_regime==regime)&(summary.objective==objective)].set_index('architecture').reindex(ARCHITECTURES)
            ax.errorbar(x,part.mean_test_ba,yerr=part.sd_test_ba.fillna(0),marker='o',capsize=4,label=objective)
        if np.isfinite(baseline_mean):
            ax.axhline(baseline_mean,linestyle='--',linewidth=1.5,label=f'Fixed250+Linear ({baseline_mean:.3f}±{baseline_sd:.3f})')
        ax.set_xticks(x,ARCHITECTURES); ax.set_xlabel('SNN architecture'); ax.set_title(regime); ax.grid(axis='y',alpha=.25)
    axes[0].set_ylabel('Test balanced accuracy'); axes[1].legend()
    fig.suptitle('Fixed250+Linear baseline vs stacked-bin SNN'); fig.tight_layout()
    fig.savefig(FIGURES_DIR/'test_ba_snn_vs_fixed250_linear.png',dpi=180,bbox_inches='tight')
    plt.show()

## Paired SNN gain over Fixed250+Linear

In [ ]:
if not runs.empty and len(baseline)==EXPECTED_BASELINES:
    paired=runs.merge(baseline[['split_seed','test_balanced_accuracy']].rename(columns={'test_balanced_accuracy':'baseline_test_ba'}),on='split_seed',how='inner')
    paired['delta_vs_fixed250_linear']=paired.test_balanced_accuracy-paired.baseline_test_ba
    paired_summary=(paired.groupby(['architecture','train_regime','objective'],as_index=False)
        .agg(mean_delta_ba=('delta_vs_fixed250_linear','mean'),sd_delta_ba=('delta_vs_fixed250_linear','std'),
             mean_snn_test_ba=('test_balanced_accuracy','mean'),mean_baseline_test_ba=('baseline_test_ba','mean')))
    display(paired_summary.sort_values('mean_delta_ba',ascending=False))
    paired.to_csv(RESULTS_DIR/'paired_snn_vs_fixed250_linear.csv',index=False)
    paired_summary.to_csv(RESULTS_DIR/'paired_snn_vs_fixed250_linear_summary.csv',index=False)

## Paired effects inside the SNN

In [ ]:
if not runs.empty:
    dyn=runs.pivot_table(index=['split_seed','architecture','objective'],columns='train_regime',values='test_balanced_accuracy',aggfunc='first').reset_index()
    if {'weight_only','trainable_dynamics'}.issubset(dyn.columns):
        dyn=dyn.dropna(subset=['weight_only','trainable_dynamics']).copy(); dyn['delta_ba']=dyn.trainable_dynamics-dyn.weight_only
        dyn_summary=dyn.groupby(['architecture','objective'],as_index=False).agg(mean_delta=('delta_ba','mean'),sd_delta=('delta_ba','std'))
        display(dyn_summary); dyn.to_csv(RESULTS_DIR/'paired_dynamics_deltas.csv',index=False)
    obj=runs.pivot_table(index=['split_seed','architecture','train_regime'],columns='objective',values='test_balanced_accuracy',aggfunc='first').reset_index()
    if {'whole_count_ce','timestep_ce'}.issubset(obj.columns):
        obj=obj.dropna(subset=['whole_count_ce','timestep_ce']).copy(); obj['delta_ba']=obj.whole_count_ce-obj.timestep_ce
        obj_summary=obj.groupby(['architecture','train_regime'],as_index=False).agg(mean_delta=('delta_ba','mean'),sd_delta=('delta_ba','std'))
        display(obj_summary); obj.to_csv(RESULTS_DIR/'paired_objective_deltas.csv',index=False)

## Learned hidden-neuron dynamics

In [ ]:
rows=[]
for p in payloads:
    if p['train_regime']!='trainable_dynamics': continue
    for h in p['best_dynamics']['hidden']:
        rows.append({'split_seed':p['split_seed'],'architecture':p['architecture'],'objective':p['objective'],'layer':h['layer'],
                     'tau_syn_ms':h['tau_syn_ms']['mean'],'tau_mem_ms':h['tau_mem_ms']['mean'],'threshold':h['threshold']['mean']})
learned=pd.DataFrame(rows)
if not learned.empty:
    learned_summary=(learned.groupby(['architecture','objective','layer'],as_index=False)
        .agg(mean_tau_syn_ms=('tau_syn_ms','mean'),sd_tau_syn_ms=('tau_syn_ms','std'),
             mean_tau_mem_ms=('tau_mem_ms','mean'),sd_tau_mem_ms=('tau_mem_ms','std'),
             mean_threshold=('threshold','mean'),sd_threshold=('threshold','std')))
    display(learned_summary); learned_summary.to_csv(RESULTS_DIR/'learned_dynamics_summary.csv',index=False)
if not runs.empty: runs.to_csv(RESULTS_DIR/'notebook_all_runs.csv',index=False)
if not baseline.empty: baseline.to_csv(RESULTS_DIR/'notebook_fixed250_linear_runs.csv',index=False)